# 04 – Sorting and Creating Columns

So far you've learned how to select data and clean missing values.

Now let's make the data more useful — by **sorting** it into a meaningful order and **creating new columns** that add information which wasn't there originally.

Creating new columns is something you'll do constantly in data work. Raw data rarely has everything you need. You almost always need to derive something — a percentage, a category, a label — from what you already have.

## Setting Up

In [ ]:
import pandas as pd
import numpy as np

data = {
    "Name":       ["Amit", "Priya", "Ravi", "Neha", "Karan", "Sneha"],
    "Marks":      [85, 90, 78, 92, 88, 74],
    "City":       ["Pune", "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi"],
    "Department": ["Science", "Arts", "Science", "Commerce", "Arts", "Commerce"],
    "Fees_Paid":  [12000, 15000, 12000, 18000, 15000, 18000]
}

df = pd.DataFrame(data)
print(df)

## 1) Sorting by a Single Column

`sort_values()` is how you sort a DataFrame. By default it sorts from lowest to highest (ascending).

In [ ]:
# Sort by Marks — lowest to highest

print(df.sort_values("Marks"))

In [ ]:
# Sort by Marks — highest to lowest

print(df.sort_values("Marks", ascending=False))

In [ ]:
# Sort alphabetically by Name

print(df.sort_values("Name"))

Just like `dropna()` and `fillna()`, `sort_values()` does not modify the original DataFrame. It returns a sorted copy. Assign it back if you want to keep the sorted order.

## 2) Sorting by Multiple Columns

When you sort by multiple columns, pandas sorts by the first column first. If two rows have the same value there, it uses the second column to break the tie.

In [ ]:
# Sort by City (A to Z), then within each city sort by Marks (high to low)

print(df.sort_values(["City", "Marks"], ascending=[True, False]))

The `ascending` parameter accepts a list when sorting by multiple columns — one `True`/`False` for each column.

Here: City sorted A→Z (`True`), Marks sorted high→low (`False`).

## 3) Sorting by Index

If you've shuffled your DataFrame or set a custom index, you can sort back by index.

In [ ]:
# First shuffle to show the point
df_shuffled = df.sample(frac=1, random_state=42)  # frac=1 means shuffle all rows
print("Shuffled:")
print(df_shuffled)
print()

# Sort back to original index order
print("Sorted by index:")
print(df_shuffled.sort_index())

## 4) Adding a New Column — Direct Assignment

The simplest way to create a new column — just assign values to a new column name.

If the column already exists, it gets overwritten. If it doesn't exist, it gets created.

In [ ]:
# Add a column from a list

df["Grade"] = ["B", "A", "C", "A", "B", "C"]
print(df)

In [ ]:
# Add a column by calculating from an existing column
# Marks are out of 100 — convert to out of 50

df["Marks_50"] = df["Marks"] / 2
print(df)

In [ ]:
# Calculate from two existing columns
# Percentage of fees paid out of a total of 20000

df["Fees_Percent"] = (df["Fees_Paid"] / 20000) * 100
print(df)

In [ ]:
# Add a constant value column — same value for every row

df["Academic_Year"] = "2024-25"
print(df)

Let's clean up the columns we added for demo purposes before moving on.

In [ ]:
df = df.drop(columns=["Marks_50", "Fees_Percent", "Academic_Year"])
print(df)

## 5) Creating Columns with `apply()`

Sometimes the logic for creating a new column is more complex than a simple math operation.

`apply()` lets you run a function on every row (or every value in a column) to produce the new column's values.

In [ ]:
# Create a Pass/Fail column based on Marks
# If Marks >= 80, the student passed. Otherwise, failed.

def pass_fail(mark):
    if mark >= 80:
        return "Pass"
    else:
        return "Fail"

df["Result"] = df["Marks"].apply(pass_fail)
print(df)

`apply(pass_fail)` runs the `pass_fail` function once for each value in the `Marks` column and collects the results into a new column.

You can do this with a regular `def` function (like above) or with a shorter `lambda` function.

In [ ]:
# Same thing using a lambda — for simple one-line logic

df["Result"] = df["Marks"].apply(lambda x: "Pass" if x >= 80 else "Fail")
print(df)

In [ ]:
# apply() on multiple conditions — assigning a proper grade

def assign_grade(mark):
    if mark >= 90:
        return "A"
    elif mark >= 80:
        return "B"
    elif mark >= 70:
        return "C"
    else:
        return "D"

df["Grade"] = df["Marks"].apply(assign_grade)
print(df)

Use `apply()` whenever the logic has conditions or is too complex to express as a single math operation.

## 6) Creating Columns with `np.where()`

`np.where()` is a clean one-liner for creating a column based on a condition.

Think of it as: **if condition → value A, else → value B**

In [ ]:
# np.where(condition, value_if_true, value_if_false)

df["Scholar"] = np.where(df["Marks"] >= 90, "Yes", "No")
print(df)

For simple True/False-style columns, `np.where()` is faster and cleaner than `apply()`. 

For more complex logic with multiple conditions, stick with `apply()`.

## 7) Renaming and Reordering Columns

Once you've added new columns, you might want to rename them or rearrange the column order.

In [ ]:
# Rename specific columns

df = df.rename(columns={"Fees_Paid": "Fees", "Scholar": "Is_Scholar"})
print(df.columns.tolist())

In [ ]:
# Reorder columns — just pass a list in the order you want

df = df[["Name", "Department", "City", "Marks", "Grade", "Result", "Is_Scholar", "Fees"]]
print(df)

Reordering columns this way is just selecting all columns in a new order — it's the same `df[[list]]` syntax you already know from selecting columns.

## 8) Dropping Columns

Sometimes you create intermediate columns just for a calculation, and you want to remove them afterwards.

In [ ]:
# Drop a single column
# axis=1 means column (axis=0 would be a row)

df = df.drop("Is_Scholar", axis=1)
print(df)

In [ ]:
# Drop multiple columns at once

df = df.drop(["Result", "Fees"], axis=1)
print(df)

## 9) Putting It All Together

Let's do a realistic example — take a raw student DataFrame and build useful new columns from scratch.

The marks are out of 150. We need percentage, grade, fees due, and a fee status flag — all derived from the three original columns.

In [ ]:
# Raw data — marks out of 150, fees paid, total fees is 20000

students = pd.DataFrame({
    "Name":      ["Amit", "Priya", "Ravi", "Neha", "Karan"],
    "Marks":     [85, 92, 67, 78, 95],
    "Fees_Paid": [20000, 12000, 20000, 15000, 8000]
})

total_fees = 20000

# 1. Percentage score — marks are out of 150
students["Percentage"] = (students["Marks"] / 150 * 100).round(1)

# 2. Grade based on marks
students["Grade"] = students["Marks"].apply(lambda x:
    "A" if x >= 90 else
    "B" if x >= 80 else
    "C" if x >= 70 else "D"
)

# 3. Fees remaining
students["Fees_Due"] = total_fees - students["Fees_Paid"]

# 4. Fee status
students["Fee_Status"] = np.where(students["Fees_Due"] == 0, "Cleared", "Pending")

# 5. Sort by Marks descending
students = students.sort_values("Marks", ascending=False).reset_index(drop=True)

print(students)

Starting from just `Name`, `Marks`, and `Fees_Paid` — we derived 4 new columns and sorted the result. This is exactly the kind of transformation you do before any reporting or analysis.

## Summary

**Sorting:**
- `df.sort_values("col")` — ascending by default
- `df.sort_values("col", ascending=False)` — descending
- `df.sort_values(["col1", "col2"], ascending=[True, False])` — multi-column sort
- `df.sort_index()` — sort back to original index order

**Creating columns:**
- `df["new"] = values` — direct assignment from a list or calculation
- `df["new"] = df["col"].apply(func)` — apply a function row by row
- `df["new"] = np.where(condition, val_if_true, val_if_false)` — fast conditional column

**Tidying up:**
- `df.rename(columns={"old": "new"})` — rename columns
- `df[[cols_in_order]]` — reorder columns
- `df.drop("col", axis=1)` — remove a column

Next up: **GroupBy** — summarising data by categories.

## Practice Questions 1

1. Create a DataFrame of 5 products with columns `Product`, `Category`, `Price`, `Units_Sold`.
2. Add a new column `Revenue` which is `Price × Units_Sold`.
3. Sort the DataFrame by `Revenue` from highest to lowest.

## Practice Questions 2

1. Add a `Price_Category` column using `apply()` — `"Expensive"` if price > 1000, `"Affordable"` otherwise.
2. Add a `Top_Seller` column using `np.where()` — `"Yes"` if `Units_Sold > 100`, `"No"` otherwise.
3. Rename `Units_Sold` to `Qty_Sold` and drop the `Price_Category` column.